In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dask

test_years = np.arange(2020,2024+1)

In [ ]:
# Load the combined station dataset
ds = xr.open_dataset("station_extracts_tasmax/all_stations_tasmax_2020-2023.nc")

print("Dataset loaded successfully!")
print(f"Dimensions: {ds.dims}")
print(f"Variables: {list(ds.data_vars)}")
print(f"\nCoordinates:")
print(f"  - Stations: {len(ds.station)}")
print(f"  - Time steps: {len(ds.time)}")
print(f"  - Date range: {ds.time.min().values} to {ds.time.max().values}")

# Create mask: keep only stations where DWD (1km) has at least one valid (non-NaN) timestep
print("\n" + "="*60)
print("Creating DWD (1km) validity mask...")
print("="*60)

dwd_has_data = ds['dwd_target_1km'].notnull().any(dim='time')
valid_stations_count = dwd_has_data.sum().values
total_stations_count = len(ds.station)

print(f"Stations with at least one valid DWD timestep: {valid_stations_count}/{total_stations_count}")
print(f"Stations being filtered out: {total_stations_count - valid_stations_count}")

# Apply mask to all variables
ds = ds.where(dwd_has_data, drop=True)

print(f"\nDataset after filtering:")
print(f"  - Stations remaining: {len(ds.station)}")

# Extract individual variables for analysis
obs = ds['observations']
era5_input = ds['era5_input_100km']
pred_10km = ds['prediction_10km']
mswx_target_10km = ds['mswx_target_10km']
pred_1km = ds['prediction_1km']
dwd_target_1km = ds['dwd_target_1km']
pred_isimip = ds['prediction_isimip_bcsd']
obs_isimip = ds['observations_isimip_bcsd']

# Create ds_obs for compatibility with plotting code
ds_obs = ds[['lat', 'lon']].assign({'station': ds.station})

print("\nVariables extracted and ready for analysis!")
print(f"Sample statistics (after DWD mask):")
print(f"  Observations mean: {obs.mean().values:.2f} mm/day")
print(f"  ERA5 input mean: {era5_input.mean().values:.2f} mm/day")
print(f"  10km prediction mean: {pred_10km.mean().values:.2f} mm/day")
print(f"  MSWX target mean: {mswx_target_10km.mean().values:.2f} mm/day")
print(f"  ISIMIP prediction mean: {pred_isimip.mean().values:.2f} mm/day")
print(f"  ISIMIP obs mean: {obs_isimip.mean().values:.2f} mm/day")
print(f"  1km prediction mean: {pred_1km.mean().values:.2f} mm/day")
print(f"  DWD target mean: {dwd_target_1km.mean().values:.2f} mm/day")

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import PowerNorm
from datasets import assign_season

# Take ensemble mean of predictions if they have ensemble dimensions
if 'ensemble' in pred_10km.dims:
    pred_10km_mean = pred_10km.mean('ensemble')
else:
    pred_10km_mean = pred_10km

if 'ensemble' in pred_1km.dims:
    pred_1km_mean = pred_1km.mean('ensemble')
else:
    pred_1km_mean = pred_1km

# ISIMIP doesn't have ensemble dimension
pred_isimip_mean = pred_isimip

# Create season labels for each dataset
obs_seasons = xr.DataArray(assign_season(obs.time), coords={'time': obs.time}, dims='time')
era5_input_seasons = xr.DataArray(assign_season(era5_input.time), coords={'time': era5_input.time}, dims='time')
pred_10km_seasons = xr.DataArray(assign_season(pred_10km_mean.time), coords={'time': pred_10km_mean.time}, dims='time')
pred_1km_seasons = xr.DataArray(assign_season(pred_1km_mean.time), coords={'time': pred_1km_mean.time}, dims='time')
mswx_10km_seasons = xr.DataArray(assign_season(mswx_target_10km.time), coords={'time': mswx_target_10km.time}, dims='time')
dwd_1km_seasons = xr.DataArray(assign_season(dwd_target_1km.time), coords={'time': dwd_target_1km.time}, dims='time')
pred_isimip_seasons = xr.DataArray(assign_season(pred_isimip_mean.time), coords={'time': pred_isimip_mean.time}, dims='time')

# Calculate raw seasonal means
obs_seasonal_raw = obs.groupby(obs_seasons).mean('time')
era5_input_seasonal_raw = era5_input.groupby(era5_input_seasons).mean('time')
pred_10km_seasonal_raw = pred_10km_mean.groupby(pred_10km_seasons).mean('time')
pred_1km_seasonal_raw = pred_1km_mean.groupby(pred_1km_seasons).mean('time')
mswx_10km_seasonal_raw = mswx_target_10km.groupby(mswx_10km_seasons).mean('time')
dwd_1km_seasonal_raw = dwd_target_1km.groupby(dwd_1km_seasons).mean('time')
pred_isimip_seasonal_raw = pred_isimip_mean.groupby(pred_isimip_seasons).mean('time')

# Create validity mask: keep only seasons with ≥80% valid observation data
print("Creating 80% valid data mask based on observations...")
valid_mask = (obs.notnull().groupby(obs_seasons).sum('time') / 
              obs_seasons.groupby(obs_seasons).count()) >= 0.8

# Apply mask to all seasonal means
obs_seasonal = obs_seasonal_raw.where(valid_mask)
era5_input_seasonal = era5_input_seasonal_raw.where(valid_mask)
pred_10km_seasonal = pred_10km_seasonal_raw.where(valid_mask)
pred_1km_seasonal = pred_1km_seasonal_raw.where(valid_mask)
mswx_10km_seasonal = mswx_10km_seasonal_raw.where(valid_mask)
dwd_1km_seasonal = dwd_1km_seasonal_raw.where(valid_mask)
pred_isimip_seasonal = pred_isimip_seasonal_raw.where(valid_mask)

# Compute all arrays
print("Computing seasonal means...")
obs_seasonal = obs_seasonal.compute()
era5_input_seasonal = era5_input_seasonal.compute()
pred_10km_seasonal = pred_10km_seasonal.compute()
pred_1km_seasonal = pred_1km_seasonal.compute()
mswx_10km_seasonal = mswx_10km_seasonal.compute()
dwd_1km_seasonal = dwd_1km_seasonal.compute()
pred_isimip_seasonal = pred_isimip_seasonal.compute()

# Print statistics about filtered data
print("\nFiltered data statistics:")
for season in ['DJF', 'MAM', 'JJA', 'SON']:
    valid_stations = valid_mask.sel(group=season).sum().values
    total_stations = len(obs.station)
    print(f"{season}: {valid_stations}/{total_stations} stations with ≥80% valid data")

In [ ]:
import cartopy
import colormaps as cmaps


# Get station coordinates from the original GHCN dataset
station_lats = ds_obs.lat.values
station_lons = ds_obs.lon.values

# Define seasons in order
seasons = ['DJF', 'MAM', 'JJA', 'SON']

# Create 4x6 panel plot (added ISIMIP column)
fig, axes = plt.subplots(4, 6, figsize=(36, 20), 
                         subplot_kw={'projection': ccrs.PlateCarree()},
                         constrained_layout=True)

# Get data for colormap range (use nonlinear scale)
all_data = np.concatenate([
    obs_seasonal.values.flatten(),
    era5_input_seasonal.values.flatten(),
    pred_10km_seasonal.values.flatten(),
    pred_isimip_seasonal.values.flatten(),
    pred_1km_seasonal.values.flatten(),
    dwd_1km_seasonal.values.flatten()
])
vmin, vmax = np.nanpercentile(all_data, [2, 98])

# Use PowerNorm for nonlinear colormap (gamma < 1 emphasizes lower values)
norm = PowerNorm(gamma=0.5, vmin=vmin, vmax=vmax)

# Column titles
col_titles = ['GHCNd', 'ERA5 (100 km)', 'Prediction (10 km)', 'ISIMIP (1 km)', 'Prediction (1 km)', 'DWD (Target-1km)']
datasets = [obs_seasonal, era5_input_seasonal, pred_10km_seasonal, pred_isimip_seasonal, pred_1km_seasonal, dwd_1km_seasonal]

# Plot each season
for row, season in enumerate(seasons):
    
    for col, (data, title) in enumerate(zip(datasets, col_titles)):
        ax = axes[row, col]
        
        # Add country boundaries
        ax.add_feature(cfeature.BORDERS, linewidth=1.5, edgecolor='black')
        ax.add_feature(cfeature.COASTLINE, linewidth=1.5, edgecolor='black')
        
        # Select seasonal data
        season_data = data.sel(group=season)
        
        # Plot stations (only non-NaN values)
        valid_mask_season = ~np.isnan(season_data.values)
        
        scatter = ax.scatter(
            station_lons[valid_mask_season],
            station_lats[valid_mask_season],
            c=season_data.values[valid_mask_season],
            s=100,
            cmap=cmaps.precip2_17lev,
            norm=norm,
            edgecolors=None,
            linewidths=0.,
            transform=ccrs.PlateCarree()
        )
        
        # Set extent based on data
        lat_buffer = (station_lats.max() - station_lats.min()) * 0.1
        lon_buffer = (station_lons.max() - station_lons.min()) * 0.1
        ax.set_extent([
            station_lons.min() - lon_buffer,
            station_lons.max() + lon_buffer,
            station_lats.min() - lat_buffer,
            station_lats.max() + lat_buffer
        ])
        
        # Add gridlines - control which panels show labels
        gl = ax.gridlines(draw_labels=True, alpha=0.3)
        gl.top_labels = False
        gl.right_labels = False
        
        # Only show left labels on leftmost column
        if col != 0:
            gl.left_labels = False
        else:
            gl.ylabel_style = {'size': 24}
        
        # Only show bottom labels on bottom row
        if row != 3:
            gl.bottom_labels = False
        else:
            gl.xlabel_style = {'size': 24}
            
        gl.xformatter = cartopy.mpl.gridliner.LONGITUDE_FORMATTER
        gl.yformatter = cartopy.mpl.gridliner.LATITUDE_FORMATTER
        gl.xlocator = plt.MultipleLocator(5)  # degrees
        gl.ylocator = plt.MultipleLocator(5)
        
        # Add titles only on top row
        if row == 0:
            ax.set_title(title, fontsize=28, fontweight='bold', pad=10)
        
        # Add season labels on left side
        if col == 0:
            ax.text(-0.35, 0.5, season, transform=ax.transAxes,
                   fontsize=28, fontweight='bold', va='center', rotation=90)

# Add colorbar on right side
fig.subplots_adjust(right=0.85)
cax = fig.add_axes([1.05, 0.15, 0.015, 0.7])
cbar = fig.colorbar(scatter, cax=cax)
cbar.set_label(f'pr (mm/day)', fontsize=20, fontweight='bold')
cbar.ax.tick_params(labelsize=24)

plt.show()

In [ ]:
from scipy.stats import spearmanr, kendalltau, wasserstein_distance

def calculate_metrics(obs_data, pred_data, wet_threshold=1.0):
    """
    Calculate multiple metrics for paired time series data at each station:
    - RMSE, Spearman ρ, Kendall τ, Wasserstein Distance
    - Frequency Bias, POD, FAR, CSI
    - Extreme Bias (95th percentile difference)
    """
    n_stations = obs_data.shape[0]
    
    metrics = {
        'rmse': np.full(n_stations, np.nan),
        'spearman': np.full(n_stations, np.nan),
        'kendall': np.full(n_stations, np.nan),
        'wasserstein': np.full(n_stations, np.nan),
        'freq_bias': np.full(n_stations, np.nan),
        'pod': np.full(n_stations, np.nan),
        'far': np.full(n_stations, np.nan),
        'csi': np.full(n_stations, np.nan),
        'extreme_bias': np.full(n_stations, np.nan)
    }
    
    for i in range(n_stations):
        obs_station = obs_data[i, :]
        pred_station = pred_data[i, :]
        
        # Find valid pairs (both obs and pred non-NaN)
        valid_mask = ~np.isnan(obs_station) & ~np.isnan(pred_station)
        
        if valid_mask.sum() < 10:  # Need at least 10 valid pairs
            continue
            
        obs_valid = obs_station[valid_mask]
        pred_valid = pred_station[valid_mask]
        
        # RMSE
        metrics['rmse'][i] = np.sqrt(np.mean((pred_valid - obs_valid)**2))
        
        # Spearman correlation
        if len(obs_valid) > 1:
            rho, _ = spearmanr(obs_valid, pred_valid)
            metrics['spearman'][i] = rho
        
        # Kendall tau
        if len(obs_valid) > 1:
            tau, _ = kendalltau(obs_valid, pred_valid)
            metrics['kendall'][i] = tau
        
        # Wasserstein distance
        metrics['wasserstein'][i] = wasserstein_distance(obs_valid, pred_valid)
        
        # Wet day detection metrics
        obs_wet = obs_valid > wet_threshold
        pred_wet = pred_valid > wet_threshold
        
        # Contingency table
        hits = np.sum(obs_wet & pred_wet)  # True positives
        misses = np.sum(obs_wet & ~pred_wet)  # False negatives
        false_alarms = np.sum(~obs_wet & pred_wet)  # False positives
        correct_negatives = np.sum(~obs_wet & ~pred_wet)  # True negatives
        
        # Frequency Bias
        obs_wet_count = np.sum(obs_wet)
        pred_wet_count = np.sum(pred_wet)
        if obs_wet_count > 0:
            metrics['freq_bias'][i] = pred_wet_count / obs_wet_count
        
        # Probability of Detection (POD) = hits / (hits + misses)
        if (hits + misses) > 0:
            metrics['pod'][i] = hits / (hits + misses)
        
        # False Alarm Ratio (FAR) = false_alarms / (hits + false_alarms)
        if (hits + false_alarms) > 0:
            metrics['far'][i] = false_alarms / (hits + false_alarms)
        
        # Critical Success Index (CSI) = hits / (hits + misses + false_alarms)
        if (hits + misses + false_alarms) > 0:
            metrics['csi'][i] = hits / (hits + misses + false_alarms)
        
        # Extreme Bias (95th percentile difference)
        if len(obs_valid) > 80:  # Need enough data for percentiles
            p95_obs = np.percentile(obs_valid, 90)
            p95_pred = np.percentile(pred_valid, 90)
            metrics['extreme_bias'][i] = (p95_pred - p95_obs)
    
    return metrics

# Align all datasets to common time steps before calculating metrics
print("Aligning datasets to common time steps...")
# Find common time steps across all datasets
common_times = obs.time.values

# Align each model dataset to observation times and transpose to match (station, time)
era5_input_aligned = era5_input.sel(time=common_times, method='nearest').transpose('station', 'time')
pred_10km_aligned = pred_10km_mean.sel(time=common_times, method='nearest').transpose('station', 'time')
pred_isimip_aligned = pred_isimip_mean.sel(time=common_times, method='nearest').transpose('station', 'time')
mswx_target_aligned = mswx_target_10km.sel(time=common_times, method='nearest').transpose('station', 'time')
pred_1km_aligned = pred_1km_mean.sel(time=common_times, method='nearest').transpose('station', 'time')
dwd_target_aligned = dwd_target_1km.sel(time=common_times, method='nearest').transpose('station', 'time')

print(f"  Observations shape: {obs.shape}")
print(f"  ERA5 aligned shape: {era5_input_aligned.shape}")
print(f"  Prediction (10km) aligned shape: {pred_10km_aligned.shape}")
print(f"  ISIMIP aligned shape: {pred_isimip_aligned.shape}")
print(f"  MSWX target aligned shape: {mswx_target_aligned.shape}")
print(f"  Prediction (1km) aligned shape: {pred_1km_aligned.shape}")
print(f"  DWD target aligned shape: {dwd_target_aligned.shape}")

# Calculate metrics for each model
print("\nCalculating metrics for ERA5 input (100 km)...")
metrics_era5 = calculate_metrics(obs.values, era5_input_aligned.values)

print("Calculating metrics for 10km prediction...")
metrics_10km = calculate_metrics(obs.values, pred_10km_aligned.values)

print("Calculating metrics for ISIMIP (10km)...")
metrics_isimip = calculate_metrics(obs.values, pred_isimip_aligned.values)

print("Calculating metrics for MSWX target (10km)...")
metrics_mswx = calculate_metrics(obs.values, mswx_target_aligned.values)

print("Calculating metrics for 1km prediction...")
metrics_1km = calculate_metrics(obs.values, pred_1km_aligned.values)

print("Calculating metrics for DWD target (1km)...")
metrics_dwd = calculate_metrics(obs.values, dwd_target_aligned.values)

print("\nMetrics calculation complete!")
print(f"Valid stations with metrics:")
print(f"  ERA5: {np.sum(~np.isnan(metrics_era5['rmse']))} stations")
print(f"  Prediction (10km): {np.sum(~np.isnan(metrics_10km['rmse']))} stations")
print(f"  ISIMIP: {np.sum(~np.isnan(metrics_isimip['rmse']))} stations")
print(f"  MSWX Target: {np.sum(~np.isnan(metrics_mswx['rmse']))} stations")
print(f"  Prediction (1km): {np.sum(~np.isnan(metrics_1km['rmse']))} stations")
print(f"  DWD Target: {np.sum(~np.isnan(metrics_dwd['rmse']))} stations")

In [ ]:
# Get station coordinates
station_lats = ds_obs.lat.values
station_lons = ds_obs.lon.values

# Define all metrics
all_metric_info = [
    {'name': 'RMSE (mm/day)', 'key': 'rmse', 'cmap': cmaps.agsunset_r, 'center': None},
    {'name': 'Spearman ρ', 'key': 'spearman', 'cmap': cmaps.agsunset_r, 'center': None, 'vmin': 0.6, 'vmax': 0.95},
    {'name': 'Kendall τ', 'key': 'kendall', 'cmap': cmaps.agsunset_r, 'center': None, 'vmin': 0.4, 'vmax': 0.8},
    {'name': 'Wasserstein Distance (mm/day)', 'key': 'wasserstein', 'cmap': cmaps.agsunset_r, 'center': None},
    {'name': 'Frequency Bias', 'key': 'freq_bias', 'cmap': cmaps.temps, 'center': 1.0, 'vmin': 0.5, 'vmax': 1.5},
    {'name': 'Probability of Detection (POD)', 'key': 'pod', 'cmap': cmaps.agsunset_r, 'center': None, 'vmin': 0.5, 'vmax': 1},
    {'name': 'False Alarm Ratio (FAR)', 'key': 'far', 'cmap': 'RdYlGn_r', 'center': None, 'vmin': 0.5, 'vmax': 1},
    {'name': 'Critical Success Index (CSI)', 'key': 'csi', 'cmap': cmaps.agsunset_r, 'center': None, 'vmin': 0.5, 'vmax': 0.8},
    {'name': 'Extreme Bias (mm/day)', 'key': 'extreme_bias', 'cmap': cmaps.precip_diff_12lev, 'center': 0.}
]

# Order: ERA5, Prediction (10 km), ISIMIP (10 km), MSWX, Prediction (1km), DWD
model_names = ['ERA5 (100 km)', 'Prediction (10 km)', 'ISIMIP (1 km)', 'Prediction (1 km)']#, , 'MSWX (10 km)', 'DWD (1 km)']
all_metrics = [metrics_era5, metrics_10km, metrics_isimip, metrics_1km]#, metrics_mswx, metrics_dwd]

# Create a separate figure for each metric
for metric_info in all_metric_info:
    metric_name = metric_info['name']
    metric_key = metric_info['key']
    
    print(f"\nCreating figure for {metric_name}...")
    
    # Create figure with 1 row x 6 columns (one for each model)
    fig, axes = plt.subplots(2, 2, figsize=(20, 16),
                             subplot_kw={'projection': ccrs.PlateCarree()},
                             constrained_layout=True)
    
    # Collect all values across models to determine colorbar range
    all_values = []
    for metrics in all_metrics:
        all_values.extend(metrics[metric_key][~np.isnan(metrics[metric_key])])
    
    # Determine colormap range
    if 'vmin' in metric_info and 'vmax' in metric_info:
        vmin, vmax = metric_info['vmin'], metric_info['vmax']
    elif metric_info['center'] is not None:
        # For centered colormaps, ensure symmetric range
        max_abs = max(abs(np.nanpercentile(all_values, 5) - metric_info['center']),
                     abs(np.nanpercentile(all_values, 95) - metric_info['center']))
        vmin = metric_info['center'] - max_abs
        vmax = metric_info['center'] + max_abs
    else:
        vmin, vmax = np.nanpercentile(all_values, [5, 95])
    
    # Plot each model
    for col, (model_name, metrics) in enumerate(zip(model_names, all_metrics)):
        ax = axes.flatten()[col]
        
        # Add geographic features
        ax.add_feature(cfeature.BORDERS, linewidth=1.0, edgecolor='black')
        ax.add_feature(cfeature.COASTLINE, linewidth=1.0, edgecolor='black')
        
        # Get metric values
        metric_values = metrics[metric_key]
        valid_mask = ~np.isnan(metric_values)
        
        if valid_mask.sum() == 0:
            ax.text(0.5, 0.5, 'No valid data', transform=ax.transAxes,
                   ha='center', va='center', fontsize=16)
            continue
        
        # Plot stations
        scatter = ax.scatter(
            station_lons[valid_mask],
            station_lats[valid_mask],
            c=metric_values[valid_mask],
            s=350,
            cmap=metric_info['cmap'],
            vmin=vmin,
            vmax=vmax,
            edgecolors='black',
            linewidths=0.,
            transform=ccrs.PlateCarree()
        )
        
        # Set extent
        lat_buffer = (station_lats.max() - station_lats.min()) * 0.1
        lon_buffer = (station_lons.max() - station_lons.min()) * 0.1
        ax.set_extent([
            station_lons.min() - lon_buffer,
            station_lons.max() + lon_buffer,
            station_lats.min() - lat_buffer,
            station_lats.max() + lat_buffer
        ])
        
        # Add gridlines
        gl = ax.gridlines(draw_labels=True, alpha=0.3)
        gl.top_labels = False
        gl.right_labels = False
        
        # if col >1:
        #     gl.left_labels = False
        # else:
        #     gl.ylabel_style = {'size': 18}
        gl.left_labels = True
        gl.bottom_labels = True
        gl.xlabel_style = {'size': 18}
        gl.ylabel_style = {'size': 18}
        
        gl.xformatter = cartopy.mpl.gridliner.LONGITUDE_FORMATTER
        gl.yformatter = cartopy.mpl.gridliner.LATITUDE_FORMATTER
        gl.xlocator = plt.MultipleLocator(5)
        gl.ylocator = plt.MultipleLocator(5)
        
        # Add model name as title
        ax.set_title(model_name, fontsize=24, fontweight='bold', pad=10)
        
        # Add summary statistics
        mean_val = np.nanmean(metric_values)
        median_val = np.nanmedian(metric_values)
        ax.text(0.02, 0.98, f'Mean: {mean_val:.3f}\nMedian: {median_val:.3f}',
               transform=ax.transAxes, fontsize=24, va='top',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Add a single colorbar for all subplots
    fig.subplots_adjust(right=0.92)
    cax = fig.add_axes([1.05, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(scatter, cax=cax)
    cbar.set_label(metric_name, fontsize=20, fontweight='bold')
    cbar.ax.tick_params(labelsize=20)
    
    # Add overall figure title
    fig.suptitle(f'{metric_name}', fontsize=28, fontweight='bold', y=1.02)
    
    plt.show()

In [ ]:
# Quantile Difference Plot: Compare model vs GHCNd observations
from scipy.stats import pearsonr

def calculate_quantile_diff(obs_data, model_data, n_quantiles=60):
    """
    Calculate quantile difference data (model - obs) at each quantile level.
    Returns (quantile_levels, quantile_differences) after removing NaN values.
    """
    # Flatten and remove NaN values
    obs_flat = obs_data.values.flatten()
    model_flat = model_data.values.flatten()
    
    # Create mask for valid pairs (both non-NaN)
    valid_mask = ~np.isnan(obs_flat) & ~np.isnan(model_flat) & (obs_flat > .1)
    
    obs_valid = obs_flat[valid_mask]
    model_valid = model_flat[valid_mask]
    
    if len(obs_valid) < 100:
        return None, None
    
    # Calculate quantiles
    quantile_levels = np.linspace(5, 95, n_quantiles)
    obs_quantiles = np.percentile(obs_valid, quantile_levels)
    model_quantiles = np.percentile(model_valid, quantile_levels)
    
    # Calculate difference
    quantile_diff = (model_quantiles - obs_quantiles)
    
    return quantile_levels, quantile_diff, obs_quantiles

# Prepare model data - similar colors for predictions, ISIMIP different, MSWX/DWD gray
model_data_list = [
    ('ERA5 (100 km)', era5_input_aligned, 'o', '#8c564b'),       # Brown
    ('Prediction (10 km)', pred_10km_aligned, 'o', '#1f77b4'),  # Blue
    ('Prediction (1 km)', pred_1km_aligned, 's', '#5a9bd4'),    # Lighter blue
    ('ISIMIP (1 km)', pred_isimip_aligned, '^', '#2ca02c'),     # Green (different)
    ('MSWX (10 km)', mswx_target_aligned, 'D', '#7f7f7f'),      # Gray
    ('DWD (1 km)', dwd_target_aligned, 'v', '#505050'),         # Darker gray
]

# Create single plot
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

# Store obs_quantiles for x-axis labels (from first model)
obs_quantile_values = None

# Plot all models on same axes
for idx, (model_name, model_data, marker, color) in enumerate(model_data_list):
    # Calculate quantile difference data
    q_levels, q_diff, obs_q = calculate_quantile_diff(obs, model_data, n_quantiles=60)
    
    if q_levels is None:
        print(f"Skipping {model_name} - insufficient data")
        continue
    
    # Store obs quantiles from first successful model for axis labels
    if obs_quantile_values is None:
        obs_quantile_values = obs_q
        quantile_levels = q_levels
    
    # Plot quantile difference
    ax.plot(q_levels, q_diff, linestyle='-', linewidth=2, marker=marker,
           color=color, label=model_name, alpha=0.8, markersize=6, markevery=3)

# Add zero reference line
ax.axhline(y=0, color='k', linestyle='--', linewidth=2, label='Perfect agreement', zorder=0)

# Add grid
ax.grid(True, alpha=0.3, zorder=0)

# Labels
ax.set_xlabel('Quantile (%) with observed values [deg C]', fontsize=18, fontweight='bold', labelpad=35)
ax.set_ylabel('Model - Observed (deg C)', fontsize=18, fontweight='bold')
ax.set_title('Quantile Difference: Models vs GHCNd Observations', fontsize=22, fontweight='bold', pad=15)

# Set x-axis limits and ticks
ax.set_xlim(0, 100)
xtick_positions = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
ax.set_xticks(xtick_positions)

# Calculate observed quantile values at tick positions
obs_at_ticks = np.percentile(obs.values[~np.isnan(obs.values) & (obs.values > 0.1)], xtick_positions)

# Set tick labels with percentile values
ax.set_xticklabels([f'{int(p)}' for p in xtick_positions])

# Add second row of labels with observed quantile values
for pos, val in zip(xtick_positions, obs_at_ticks):
    ax.text(pos, -0.05, f'[{val:.1f}]', 
            ha='center', va='top', fontsize=12, 
            transform=ax.get_xaxis_transform(), 
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='none', alpha=0.7))

# Add legend
ax.legend(loc='lower left', fontsize=20, framealpha=0.9)

# Increase tick label size
ax.tick_params(axis='both', which='major', labelsize=16)

plt.tight_layout()
plt.show()

print("\nQuantile Difference Plot Summary:")
print("Positive values indicate model overestimation at those quantiles")
print("Negative values indicate model underestimation at those quantiles")
print("Zero line represents perfect agreement between model and observations")
print("Values in brackets show observed precipitation values at each quantile")
print("Blue shades: Predictions (10km and 1km)")
print("Green: ISIMIP")
print("Gray shades: MSWX and DWD targets")

In [ ]:
# 2D Density Plot: Model vs GHCNd Observations (Station Mean Values)
from scipy.stats import pearsonr
import seaborn as sns

def calculate_metrics_and_means(obs_data, model_data):
    """
    Calculate performance metrics on raw time series data, then return time means for plotting.
    Returns: obs_mean, model_mean, and metrics dictionary
    """
    # Flatten and remove NaN values for metric calculation (using all time steps)
    obs_flat = obs_data.values.flatten()
    model_flat = model_data.values.flatten()
    
    # Create mask for valid pairs (both non-NaN and obs > 0.1)
    valid_mask = ~np.isnan(obs_flat) & ~np.isnan(model_flat) & (obs_flat > 0.1)
    
    obs_valid_all = obs_flat[valid_mask]
    model_valid_all = model_flat[valid_mask]
    
    if len(obs_valid_all) < 100:
        return None, None, None
    
    # Calculate metrics on RAW data (all time steps)
    # Correlation (Pearson)
    r, _ = pearsonr(obs_valid_all, model_valid_all)
    
    # R² (coefficient of determination) - proper formulation
    # Measures how well predictions match observations on 1:1 line
    SS_res = np.sum((obs_valid_all - model_valid_all)**2)  # Residual sum of squares
    SS_tot = np.sum((obs_valid_all - np.mean(obs_valid_all))**2)  # Total sum of squares
    r_squared = 1 - (SS_res / SS_tot)
    
    # Nash-Sutcliffe Efficiency (NSE)
    mean_obs = np.mean(obs_valid_all)
    nse = 1 - (np.sum((obs_valid_all - model_valid_all)**2) / np.sum((obs_valid_all - mean_obs)**2))
    
    # Percent Error (normalized by mean observed)
    percent_error = 100 * (np.mean(model_valid_all) - np.mean(obs_valid_all)) / np.mean(obs_valid_all)
    
    metrics = {
        'r': r,
        'r2': r_squared,
        'nse': nse,
        'percent_error': percent_error
    }
    
    # Take mean across time for each station FOR PLOTTING
    obs_mean = np.nanmean(obs_data.values, axis=1)  # Shape: (n_stations,)
    model_mean = np.nanmean(model_data.values, axis=1)  # Shape: (n_stations,)
    
    # Create mask for valid station pairs (both non-NaN)
    valid_mask_stations = ~np.isnan(obs_mean) & ~np.isnan(model_mean) & (obs_mean > 0.1)
    
    obs_valid = obs_mean[valid_mask_stations]
    model_valid = model_mean[valid_mask_stations]
    
    return obs_valid, model_valid, metrics

# Prepare model data with colors
model_data_list = [
    ('ERA5 (100 km)', era5_input_aligned, '#8c564b'),       # Brown
    ('Prediction (10 km)', pred_10km_aligned, '#1f77b4'),  # Blue
    ('Prediction (1 km)', pred_1km_aligned, '#5a9bd4'),    # Lighter blue
    ('ISIMIP (1 km)', pred_isimip_aligned, '#2ca02c'),     # Green
    ('MSWX (10 km)', mswx_target_aligned, '#7f7f7f'),      # Gray
    ('DWD (1 km)', dwd_target_aligned, '#505050'),         # Darker gray
]

# Set seaborn style
sns.set_theme(style="whitegrid")

# Create figure with 2x3 subplots
fig, axes = plt.subplots(2, 3, figsize=(20, 14), sharex=True, sharey=True)

# Find global min/max for consistent axis limits (using mean values)
obs_station_means = np.nanmean(obs.values, axis=1)
obs_valid_means = obs_station_means[~np.isnan(obs_station_means) & (obs_station_means > 0.1)]
global_max = np.percentile(obs_valid_means, 99.5)

# Plot each model
for idx, (model_name, model_data, color) in enumerate(model_data_list):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    # Calculate metrics on raw data, get time means for plotting
    obs_valid, model_valid, metrics = calculate_metrics_and_means(obs, model_data)
    
    if obs_valid is None:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes,
               ha='center', va='center', fontsize=16)
        ax.set_title(model_name, fontsize=18, fontweight='bold')
        continue
    
    # Draw scatter plot (all points - only ~500 stations)
    sns.scatterplot(x=obs_valid, y=model_valid, s=30, color="0", alpha=0.5, ax=ax, legend=False)
    
    # Draw 2D histogram with white-to-red colormap
    sns.histplot(x=obs_valid, y=model_valid, bins=30, pthresh=0.05, cmap="Reds", 
                 ax=ax, cbar=False, alpha=0.8)
    
    # Add 1:1 line
    ax.plot([0, global_max], [0, global_max], 'k--', linewidth=2.5, 
            label='1:1 line', zorder=10, alpha=0.9)
    
    # Set limits
    ax.set_xlim(0, global_max)
    ax.set_ylim(0, global_max)
    
    # Labels
    ax.set_xlabel('GHCNd Observed (mm/day)', fontsize=20, fontweight='bold')
    ax.set_ylabel('Model (mm/day)', fontsize=20, fontweight='bold')
    ax.set_title(model_name, fontsize=18, fontweight='bold', pad=10)
    
    # Add grid
    ax.grid(True, alpha=0.3, zorder=0)
    
    # Add metrics text box (metrics from RAW data)
    metrics_text = f"r = {metrics['r']:.3f}\n"
    metrics_text += f"R² = {metrics['r2']:.3f}\n"
    metrics_text += f"NSE = {metrics['nse']:.3f}\n"
    metrics_text += f"Error = {metrics['percent_error']:.1f}%\n"
    # metrics_text += f"N = {len(obs_valid)} stations"
    
    ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes,
           fontsize=20, va='top', ha='left',
           bbox=dict(boxstyle='round', facecolor='white', edgecolor='black', 
                    alpha=0.9, linewidth=1.5))
    
    # Equal aspect ratio
    ax.set_aspect('equal', adjustable='box')
    
    # Tick parameters
    ax.tick_params(axis='both', which='major', labelsize=22)

# Add overall title
fig.suptitle('2D Density: Station Mean Precipitation - Model vs GHCNd Observations', 
            fontsize=24, fontweight='bold', y=0.995)

plt.tight_layout()
plt.show()

# Reset seaborn style
sns.reset_defaults()

print("\n2D Density Plot Summary:")
print("Each point represents one station's mean precipitation across all time steps")
print("Scatter points show individual station mean values")
print("Heat map shows density of station means (white = low density, red = high density)")
print("Black dashed line: Perfect 1:1 agreement")
print("Metrics displayed (calculated on raw daily data):")
print("  - r: Pearson correlation coefficient")
print("  - R²: Coefficient of determination (1 - SS_res/SS_tot)")
print("  - NSE: Nash-Sutcliffe Efficiency (1 = perfect, 0 = mean predictor)")
print("  - Error: Mean percent bias")
print("  - N: Number of stations with valid data")


In [ ]:
obs